# AL-MIZAN: membuat embedding Dense di Kaggle (GPU)

Notebook ini menjalankan `build_index.py` yang sama dengan di laptop, tetapi Ollama memakai GPU Kaggle.
Hasilnya 3 file `.npy` yang diunduh ke folder `embeddings/` proyek di laptop.

**Pengaturan notebook (panel kanan → Settings):**
- Accelerator: **GPU T4 x2** atau **GPU P100**
- Internet: **On** (butuh akun Kaggle yang sudah verifikasi nomor HP)

**Input:** dataset Kaggle berisi isi `almizan_kaggle_data.zip`
(`data/islamqa.csv`, `data/quran.csv`, `data/hadith.csv`, `islamqa_rag/`, `build_index.py`).

## 1. Salin data & kode ke folder kerja

In [ ]:
import glob
import os
import shutil
import subprocess
import sys
import time
import urllib.request
import zipfile

WORK = "/kaggle/working/almizan"

# Kaggle biasanya mengekstrak zip saat upload; jika belum, ekstrak di sini
found = glob.glob("/kaggle/input/**/build_index.py", recursive=True)
if found:
    shutil.copytree(os.path.dirname(found[0]), WORK, dirs_exist_ok=True)
else:
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    assert zips, "Dataset belum ditambahkan: klik 'Add Input' lalu pilih dataset berisi almizan_kaggle_data.zip"
    zipfile.ZipFile(zips[0]).extractall(WORK)

print("Isi folder kerja:", sorted(os.listdir(WORK)))
print("Data:", sorted(os.listdir(f"{WORK}/data")))

## 2. Cek GPU

In [ ]:
!nvidia-smi

## 3. Install Ollama & library Python

`zstd` dibutuhkan installer Ollama untuk mengekstrak paketnya, `pciutils` agar installer mendeteksi GPU NVIDIA.

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install ollama rank_bm25

## 4. Menyalakan server Ollama & mengunduh model embedding

In [ ]:
server = subprocess.Popen(["ollama", "serve"], stdout=open("/kaggle/working/ollama.log", "w"),
                          stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print("Ollama siap:", urllib.request.urlopen("http://127.0.0.1:11434/api/version").read().decode())
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Server Ollama tidak menyala, lihat /kaggle/working/ollama.log")

In [ ]:
!ollama pull nomic-embed-text-v2-moe

## 5. Uji kecepatan & pastikan GPU dipakai

Kolom `PROCESSOR` pada `ollama ps` harus menunjukkan **GPU**. Jika tertulis CPU, cek pengaturan Accelerator.

In [ ]:
sys.path.insert(0, WORK)
from islamqa_rag.data import load_corpora
from islamqa_rag.retrievers import embed

corpora = load_corpora()
total = sum(len(df) for df in corpora.values())
sample = corpora["hadith"]["search_text"].tolist()[:256]

start = time.time()
vectors = embed(sample, "search_document: ")
seconds = time.time() - start
print(f"{vectors.shape} dalam {seconds:.1f} detik -> perkiraan total {seconds / 256 * total / 60:.0f} menit "
      f"untuk {total:,} dokumen")

In [ ]:
!ollama ps

## 6. Membuat embedding semua corpus

In [ ]:
!cd {WORK} && python build_index.py quran islamqa hadith

## 7. Menyiapkan file untuk diunduh

Hasil ada di `/kaggle/working/embeddings/` dan juga dalam bentuk zip `embeddings_almizan.zip`.
Unduh dari panel **Output** (kanan), lalu taruh ketiga file `.npy` di folder `embeddings/` proyek di laptop.

In [ ]:
OUT = "/kaggle/working/embeddings"
os.makedirs(OUT, exist_ok=True)
for f in sorted(glob.glob(f"{WORK}/embeddings/*.npy")):
    shutil.copy(f, OUT)
    print(f"{os.path.basename(f):<45} {os.path.getsize(f) / 1e6:7.1f} MB")

shutil.make_archive("/kaggle/working/embeddings_almizan", "zip", OUT)
shutil.rmtree(WORK)          # data & kode tidak perlu ikut di output
print("\nSiap diunduh:", sorted(os.listdir("/kaggle/working")))